In [1]:
# Load and combine raw dataset for cleaning
import pandas as pd

file_path = "../data/raw/online_retail_II.xlsx"

df_0910 = pd.read_excel(file_path, sheet_name="Year 2009-2010")

df_1011 =  pd.read_excel(file_path, sheet_name="Year 2010-2011")

df = pd.concat([df_0910, df_1011], ignore_index=True)

print("Initial shape:", df.shape)

Initial shape: (1067371, 8)


In [2]:
# Store current row count before removing anything
before = len(df)

 # # Drop any rows that are completely identical across all columns
df = df.drop_duplicates()

after = len(df)

print ("Duplicates removed:", before - after)
print("New shape:", df.shape)

Duplicates removed: 34335
New shape: (1033036, 8)


In [ ]:
# Store current row count
before = len(df)

# Pull out rows where Invoice starts with 'C' - these are cancellations
df_cancelled =  df[df['Invoice'].astype(str).str.startswith('C')]

# Keep only rows where Invoice does NOT start with 'C'
df =  df[~df['Invoice'].astype(str).str.startswith('C')]

# Store row count after removal 
after  = len(df)

print("Cancelled rows separated:", before - after)

print("Remaining data shape:", df.shape)
print("Cancelled dataset shape:", df_cancelled.shape)

Cancelled rows separated: 19104
Remaining data shape: (1013932, 8)
Cancelled dataset shape: (19104, 8)


In [ ]:
# Store current row count 
before =  len(df)

# Keep only rows where Quantity is greater than 0 - removes returns/reversals
df = df[df['Quantity'] > 0]

# Store row count after removal
after = len(df)

print("Negative quantity rows removed:", before - after)

print("New shape:", df.shape)

Negative quantity rows removed: 3393
New shape: (1010539, 8)


In [5]:
# Store current row count
before = len(df)

# Keep only rows where Price is greater than 0 - removes zero and negative prices
df = df[df['Price'] > 0]

# Store row count after removal
after = len(df)

print ("Invalid price rows removed:", before - after)

print("New shape:", df.shape)

Invalid price rows removed: 2626
New shape: (1007913, 8)


In [6]:

before = len(df)

# Drop any row where Customer ID is blank (required for RFM analysis)
df = df.dropna(subset=['Customer ID'])

# Store row count after removal
after =  len(df)

print("Rows without Customer ID removed:", before - after)

print("New shape:", df.shape)

Rows without Customer ID removed: 228488
New shape: (779425, 8)


In [7]:
# Create revenue feature (Quantity × Price)

# Multiply Quantity by Price for each row to get revenue per line item
df['Revenue'] = df['Quantity'] * df['Price']

print("Revenue column added.")

Revenue column added.


In [8]:
# Convert Customer ID to integer after cleaning (Safe to do now since all nulls have been removed)

# Customer ID was stored as float  convert to int 
df['Customer ID'] = df['Customer ID'].astype(int)

In [9]:

# After dropping rows the index has gaps 

# Reset it to be clean and sequential starting from 0
df = df.reset_index(drop=True)

In [10]:
# Confirm final row and column count after all cleaning steps 
print("Final shape:", df.shape)

# Check every column has zero missing values
print ("\nMissing values:")
print(df.isnull().sum())

# Confirm no duplicate rows remain
print ("\nDuplicate rows:", df.duplicated().sum())

# # Confirm no negative or zero values slipped through
print("\nMin Quantity:", df['Quantity'].min())

print("Min Price:", df['Price'].min())

Final shape: (779425, 9)

Missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
dtype: int64

Duplicate rows: 0

Min Quantity: 1
Min Price: 0.001


#Observation:Although missing Description values were not explicitly removed, these rows were eliminated indirectly during earlier cleaning steps (such as removing rows with missing Customer ID and invalid transactions).

In [11]:
# # Save cleaned transactions to CSV 
df.to_csv("../data/processed/cleaned_data.csv", index=False)

# Save cancellations separately (preserved for potential returns analysis)
df_cancelled.to_csv("../data/processed/cancelled_data.csv", index=False)

In [12]:
#  Show first 5 rows of cleaned data 
print("Cleaned dataset preview:")
display (df.head())


# Show first 5 rows of cancellations 
print("\nCancelled dataset preview:")
display (df_cancelled.head())

# Show shapes (size) of dataset 
print("\nCleaned dataset shape:", df.shape)
print("Cancelled dataset shape:", df_cancelled.shape)

Cleaned dataset preview:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0



Cancelled dataset preview:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia



Cleaned dataset shape: (779425, 9)
Cancelled dataset shape: (19104, 8)
